# Linux user and group management

Try the command preview for a lab account, then check which names the script accepts. The cells only use preview mode.

Use **Runtime → Run all** in Google Colab. Each run uses a fresh temporary lab directory. The output below was captured on Linux.

[View the project on GitHub](https://github.com/Sehaj64/Portfolio/tree/main/projects/linux-operations/user-management)

## Prepare the lab

In [1]:
%%bash
set -eu
lab=$(mktemp -d)
printf '%s' "$lab" > user-management.lab
cd "$lab"
cat > users.sh <<'BASH_FILE'
#!/usr/bin/env bash
# Preview by default. Mutations limited to opslab_ names on a disposable Linux lab.
set -euo pipefail
umask 077
apply=0
if [[ ${1:-} == --apply ]]; then apply=1; shift; fi
[[ $# == 2 || $# == 3 ]] || { echo "Usage: $0 [--apply] {create|group|add-group|lock|unlock|inspect} opslab_NAME [opslab_GROUP]" >&2; exit 1; }
action=$1; name=$2; group=${3:-}
valid() { [[ $1 =~ ^opslab_[a-z][a-z0-9_]{0,20}$ ]]; }
valid "$name" || { echo 'Names must start opslab_ and use lowercase letters, numbers or underscores' >&2; exit 1; }
case "$action" in
 create) cmd=(useradd --create-home --shell /bin/bash -- "$name");;
 group) cmd=(groupadd -- "$name");;
 add-group) valid "$group" || { echo 'Valid opslab_ group required' >&2; exit 1; }; cmd=(usermod -aG "$group" -- "$name");;
 lock) cmd=(usermod -L -- "$name");;
 unlock) cmd=(usermod -U -- "$name");;
 inspect) id -- "$name"; exit;;
 *) echo 'Unknown action' >&2; exit 1;;
esac
if [[ $action != add-group && $# != 2 ]]; then echo 'Unexpected group argument' >&2; exit 1; fi
if (( !apply )); then printf 'PLAN '; printf '%q ' "${cmd[@]}"; printf '\n'; exit; fi
(( EUID == 0 )) || { echo 'Apply requires root in a disposable lab' >&2; exit 1; }
if [[ $action == create ]] && getent passwd "$name" > /dev/null; then echo 'EXISTS user'; exit; fi
if [[ $action == group ]] && getent group "$name" > /dev/null; then echo 'EXISTS group'; exit; fi
if [[ $action == add-group ]]; then getent group "$group" > /dev/null || { echo 'Group missing' >&2; exit 1; }; fi
if [[ $action != create && $action != group ]]; then
  entry=$(getent passwd "$name") || { echo 'User missing' >&2; exit 1; }
  IFS=: read -r _ _ uid _ <<< "$entry"
  (( uid >= 1000 && uid != 65534 )) || { echo 'Refusing a system account' >&2; exit 1; }
fi
"${cmd[@]}"
printf 'APPLIED action=%s name=%s\n' "$action" "$name"
BASH_FILE
echo 'Lab scripts ready.'

Lab scripts ready.


## Preview account creation and group membership

In [2]:
%%bash
set -eu
cd "$(cat user-management.lab)"
bash users.sh create opslab_demo
bash users.sh group opslab_team
bash users.sh add-group opslab_demo opslab_team

PLAN useradd --create-home --shell /bin/bash -- opslab_demo 
PLAN groupadd -- opslab_team 
PLAN usermod -aG opslab_team -- opslab_demo 


## Preview a password lock and unlock

In [3]:
%%bash
set -eu
cd "$(cat user-management.lab)"
bash users.sh lock opslab_demo
bash users.sh unlock opslab_demo

PLAN usermod -L -- opslab_demo 
PLAN usermod -U -- opslab_demo 


## Try an account outside the lab

In [4]:
%%bash
set -eu
cd "$(cat user-management.lab)"
if bash users.sh create root; then exit 1; fi
echo "Account name rejected before any change."

Account name rejected before any change.
Names must start opslab_ and use lowercase letters, numbers or underscores
